In [30]:
import pandas as pd, numpy as np

df = pd.read_csv('data/combined.csv')

In [31]:
df['Date_x'] = pd.to_datetime(df['Date_x']).dt.date
df['daily_change_y'] = pd.to_numeric(df['daily_change_y'], errors='coerce')
df = df.dropna(subset=['daily_change_y']).copy()

if 'Surprise(%)' in df.columns:
    df["Surprise(%)"] = pd.to_numeric(df["Surprise(%)"], errors="coerce")
    df = df.sort_values(["ticker","Date_x"])
    df["surprise_shift1"] = df.groupby("ticker")["Surprise(%)"].shift(1)

# Drop leaky columns
for col in ["Reported EPS","Surprise(%)","Adj Close"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

df["high_low_range"] = (df["High"] - df["Low"]) / df["Open"]
if "sma20" in df.columns:
    df["close_sma20_ratio"] = df["Open"] / df["sma20"]
df["vol_log"] = np.log1p(df["Volume"].fillna(0))

df['y'] = (df['daily_change_y'] >= 0.03).astype(int)

In [32]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score, precision_score, recall_score


num_feats = ['EPS Estimate', 'TTM_EPS', 'PE_Ratio', 'Forward_PE', 'Open',
             'High', 'Low', 'Volume', '1w_change', '1m_change', '3m_change',
             '1yr_change', '52wk_high', '52wk_low', 'sma20_change', 'sma50_change',
             'sma200_change', 'rsi_14', 'volatility_wk', 'volatility_1m', 
             'shifted_rep_eps', 'surprise_shift1', 'high_low_range', 'close_sma20_ratio',
             'vol_log']
num_feats = [f for f in num_feats if f in df.columns]

cat_feats = ['Quarter', 'Sector', 'Industry', 'HeadquartersState']
cat_feats = [f for f in cat_feats if f in df.columns]

df = df.sort_values("Date_x")
df = df[num_feats + cat_feats + ['y']].dropna().copy()
# y = df.pop('y')
y = df['y']

In [33]:
x_train, x_test, y_train, y_test = train_test_split(df[num_feats + cat_feats], y, test_size=0.2, shuffle=False)

# --- Outlier detection (train only) ---
outlier_feats = []
for col in num_feats:
    s = x_train[col].dropna()
    if s.empty: 
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    pct_outliers = ((s < lower) | (s > upper)).mean()
    if pct_outliers > 0.05:
        outlier_feats.append(col)

# --- Winsorization caps ---
caps = {}
for col in outlier_feats:
    caps[col] = x_train[col].quantile([0.01, 0.99]).values

def winsorize(df, caps):
    df_copy = df.copy()
    for col, (low, high) in caps.items():
        df_copy[col] = df_copy[col].clip(lower=low, upper=high)
    return df_copy

x_train = winsorize(x_train, caps)
x_test = winsorize(x_test, caps)

numeric_transform = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_transform = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer([("num", numeric_transform, num_feats),
                                  ("cat", cat_transform, cat_feats)])

In [34]:
import xgboost as xgb
xgb_params = {
    'n_estimators': 5000,
    'learning_rate': 0.01,
    'booster': 'gbtree',
    'max_depth': 6,
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'early_stopping_rounds': 250
}

models = {
    'LosgisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(n_estimators=1000, n_jobs=-1, class_weight='balanced'),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=1000),
    'xgboost': xgb.XGBClassifier(**xgb_params)
}

trained = {}
results = {}
for name, model in models.items():
    print(f'Now training: {name}')
    pipe = Pipeline([('pre', preprocessor), ('clf', model)])
    if name == 'xgboost':
        x_test_pre = preprocessor.transform(x_test)
        # Only XGBoost supports eval_set
        pipe.fit(x_train, y_train, clf__eval_set=[(x_test_pre, y_test)], clf__verbose=100)
    else:
        pipe.fit(x_train, y_train)

    prob = pipe.predict_proba(x_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    roc = roc_auc_score(y_test, prob)
    precision, recall, _ = precision_recall_curve(y_test, prob)
    pr_auc = auc(recall, precision)
    f105 = f1_score(y_test, pred)
    # Optimize threshold for F1
    thresholds = np.linspace(0.1, 0.9, 50)
    best_f1, best_thr, best_prec, best_rec = 0, 0.5, 0, 0
    for thr in thresholds:
        pred_thr = (prob >= thr).astype(int)
        f1 = f1_score(y_test, pred_thr)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
            best_prec = precision_score(y_test, pred_thr)
            best_rec = recall_score(y_test, pred_thr)
    results[name] = {
        'roc_auc': roc, 
        'pr_auc': pr_auc, 
        'f1_0.5': f105,
        'best_f1': best_f1,
        'best_threshold': best_thr,
        'prec@best': best_prec,
        'rec@best': best_rec}
    trained[name] = pipe

results

Now training: LosgisticRegression
Now training: RandomForest
Now training: GradientBoosting
Now training: xgboost
[0]	validation_0-aucpr:0.24934
[100]	validation_0-aucpr:0.26929
[200]	validation_0-aucpr:0.27243
[300]	validation_0-aucpr:0.27403
[400]	validation_0-aucpr:0.27518
[500]	validation_0-aucpr:0.27554
[600]	validation_0-aucpr:0.27470
[700]	validation_0-aucpr:0.27228
[781]	validation_0-aucpr:0.27170


{'LosgisticRegression': {'roc_auc': 0.6219343194384753,
  'pr_auc': 0.2701652834600937,
  'f1_0.5': 0.36674816625916873,
  'best_f1': 0.39118825100133514,
  'best_threshold': np.float64(0.3938775510204082),
  'prec@best': 0.2553376906318083,
  'rec@best': 0.8359486447931527},
 'RandomForest': {'roc_auc': 0.6136090333547332,
  'pr_auc': 0.26890048121253335,
  'f1_0.5': 0.0,
  'best_f1': 0.3833160978972768,
  'best_threshold': np.float64(0.1653061224489796),
  'prec@best': 0.25272727272727274,
  'rec@best': 0.7931526390870185},
 'GradientBoosting': {'roc_auc': 0.6042644885325353,
  'pr_auc': 0.2660736443630823,
  'f1_0.5': 0.06733167082294264,
  'best_f1': 0.3747211895910781,
  'best_threshold': np.float64(0.1326530612244898),
  'prec@best': 0.25339366515837103,
  'rec@best': 0.7189728958630528},
 'xgboost': {'roc_auc': 0.6267085445178134,
  'pr_auc': 0.2760204232688035,
  'f1_0.5': 0.005649717514124294,
  'best_f1': 0.3880597014925373,
  'best_threshold': np.float64(0.1653061224489796),

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from sklearn.impute import SimpleImputer
# from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# import xgboost as xgb

# # --- Load data ---
# df = pd.read_csv("data/combined.csv")
# target = "daily_change_y"
# df[target] = (df[target] >= 0.03).astype(int)

# # Features
# exclude_cols = ["ticker", "daily_change", target]
# cat_feats = [c for c in ["Sector", "Industry", "HeadquartersState"] if c in df.columns]
# num_feats = df.drop(columns=[c for c in exclude_cols + cat_feats if c in df.columns]) \
#               .select_dtypes(include=["number"]).columns.tolist()

# # --- Train/Valid split (chronological) ---
# df = df.sort_values("date") if "date" in df.columns else df
# train_df, valid_df = train_test_split(df, test_size=0.2, shuffle=False)

# # --- Outlier detection (train only) ---
# outlier_feats = []
# for col in num_feats:
#     s = train_df[col].dropna()
#     if s.empty: 
#         continue
#     q1, q3 = s.quantile([0.25, 0.75])
#     iqr = q3 - q1
#     lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
#     pct_outliers = ((s < lower) | (s > upper)).mean()
#     if pct_outliers > 0.05:
#         outlier_feats.append(col)

# # --- Winsorization caps ---
# caps = {}
# for col in outlier_feats:
#     caps[col] = train_df[col].quantile([0.01, 0.99]).values

# def winsorize(df, caps):
#     df_copy = df.copy()
#     for col, (low, high) in caps.items():
#         df_copy[col] = df_copy[col].clip(lower=low, upper=high)
#     return df_copy

# train_w = winsorize(train_df, caps)
# valid_w = winsorize(valid_df, caps)

# # --- Preprocessor ---
# num_transform = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", StandardScaler())
# ])
# cat_transform = Pipeline([
#     ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
#     ("ohe", OneHotEncoder(handle_unknown="ignore"))
# ])
# preprocessor = ColumnTransformer([
#     ("num", num_transform, num_feats),
#     ("cat", cat_transform, cat_feats)
# ])

# # --- Models ---
# models = {
#     "Logistic": LogisticRegression(max_iter=1000),
#     "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
#     "GradientBoosting": GradientBoostingClassifier(random_state=42),
# }
# # try:
# #     models["XGBoost"] = xgb.XGBClassifier(
# #         eval_metric="logloss", use_label_encoder=False, random_state=42
# #     )
# # except Exception as e:
# #     print("XGBoost not available:", e)

# # --- Evaluation helper ---
# def evaluate(model, X_train, y_train, X_valid, y_valid):
#     model.fit(X_train, y_train)
#     probs = model.predict_proba(X_valid)[:, 1]
#     preds = (probs >= 0.5).astype(int)

#     roc = roc_auc_score(y_valid, probs)
#     pr = average_precision_score(y_valid, probs)
#     f1_05 = f1_score(y_valid, preds)

#     # Optimize threshold for F1
#     thresholds = np.linspace(0.1, 0.9, 50)
#     best_f1, best_thr, best_prec, best_rec = 0, 0.5, 0, 0
#     for thr in thresholds:
#         pred_thr = (probs >= thr).astype(int)
#         f1 = f1_score(y_valid, pred_thr)
#         if f1 > best_f1:
#             best_f1 = f1
#             best_thr = thr
#             best_prec = precision_score(y_valid, pred_thr)
#             best_rec = recall_score(y_valid, pred_thr)

#     return {
#         "roc_auc": roc,
#         "pr_auc": pr,
#         "f1@0.5": f1_05,
#         "best_f1": best_f1,
#         "best_thr": best_thr,
#         "prec@best": best_prec,
#         "rec@best": best_rec,
#     }

# # --- Run experiments ---
# results = []
# for variant, (tr, va) in {
#     "baseline": (train_df, valid_df),
#     "winsorized": (train_w, valid_w)
# }.items():
#     X_train, y_train = tr.drop(columns=[target]), tr[target]
#     X_valid, y_valid = va.drop(columns=[target]), va[target]

#     for name, model in models.items():
#         pipe = Pipeline([("prep", preprocessor), ("clf", model)])
#         metrics = evaluate(pipe, X_train, y_train, X_valid, y_valid)
#         metrics["model"] = name
#         metrics["variant"] = variant
#         results.append(metrics)

# results_df = pd.DataFrame(results)
# print(results_df)
# print("Winsorized features:", outlier_feats)


c:\Users\samki\projects\earnings_data_predictions\.venv-3.10\lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['Adj Close']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\samki\projects\earnings_data_predictions\.venv-3.10\lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['Adj Close']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\samki\projects\earnings_data_predictions\.venv-3.10\lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['Adj Close']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\samki\projects\earnings_data_predictions\.venv-3.10\lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed 

    roc_auc    pr_auc    f1@0.5   best_f1  best_thr  prec@best  rec@best  \
0  0.554890  0.240389  0.012959  0.369308  0.148980   0.238699  0.815556   
1  0.640930  0.317522  0.019651  0.406750  0.181633   0.275522  0.776667   
2  0.645707  0.320342  0.042463  0.406345  0.197959   0.291808  0.668889   
3  0.580237  0.264273  0.029258  0.380755  0.148980   0.248146  0.817778   
4  0.640934  0.314313  0.015317  0.409938  0.197959   0.284483  0.733333   
5  0.644985  0.323349  0.052521  0.405251  0.197959   0.290681  0.668889   

              model     variant  
0          Logistic    baseline  
1      RandomForest    baseline  
2  GradientBoosting    baseline  
3          Logistic  winsorized  
4      RandomForest  winsorized  
5  GradientBoosting  winsorized  
Winsorized features: ['EPS Estimate', 'Reported EPS', 'Surprise(%)', 'Close', 'High', 'Low', 'Open', 'Volume', 'daily_change_x', '1yr_change', '52wk_high', '52wk_low', 'sma20', 'sma50', 'sma200', 'volatility_wk', 'volatility_1m',